# OmniScore Evaluation on XSAMSum (English → Chinese)

This notebook evaluates Chinese summaries on part of the XSAMSum dataset using
**OmniScore** (Alam, Bhatia, Laskar, Chowdhury 2026), a deterministic encoder-based
multilingual evaluator that scores generated text on four dimensions:
**informativeness**, **clarity**, **plausibility**, **faithfulness**.
Scores are continuous in `[1, 5]`.

We use the [`QCRI/OmniScore-deberta-v3`](https://huggingface.co/QCRI/OmniScore-deberta-v3)
checkpoint (0.2B parameters, DeBERTa-v3-base backbone, custom regression head).

## What this notebook does

XSAMSum is the Chinese-translation extension of SAMSum (chit-chat dialogue
summarization). Each example contains:

- An **English dialogue** (the source)
- A **Chinese reference summary** (gold)
- A **Chinese candidate summary** (the system output we want to score)

We score each candidate in **two modes**:

| Mode | Inputs to OmniScore | What it tells us |
|------|---------------------|------------------|
| Source-grounded | `Source = English dialogue`, `Candidate = Chinese summary` | Faithfulness to the English source (catches cross-lingual hallucination) |
| Reference-based | `Reference = Chinese gold`, `Candidate = Chinese summary` | Closeness to the human-written Chinese summary |

For each mode we get four scalar scores per example, then aggregate to corpus level.

## Layout

1. Setup & dependencies
2. Load OmniScore
3. Build / load the XSAMSum slice
4. Format inputs for OmniScore
5. Run scoring (both modes, batched)
6. Aggregate and report
7. Save results to disk

## Important caveats

- OmniScore was trained on 107 languages but Chinese-specific correlation
  numbers are not separately reported in the paper. Treat these scores as a
  *signal*, not ground truth.
- The model truncates inputs at 512 tokens. SAMSum dialogues can be longer
  than that once formatted; we report truncation rates so you know how much
  of each example actually gets scored.
- These scores are not directly comparable to ROUGE/BERTScore or to ClidSum's
  human-study axes (grammaticality / informativeness / conciseness).
  OmniScore extends the evaluation; it does not replace it.

## 1. Setup & dependencies

In [ ]:
!pip uninstall -y torchvision torchaudio

In [ ]:
# Install required packages (uncomment if running fresh)
!pip install -q -U torch transformers sentencepiece datasets pandas tqdm

In [ ]:
import os
import json
import math
import time
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional

import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

# Reproducibility — OmniScore is deterministic at inference, but we still
# fix seeds so that any sampling we do (e.g. selecting a slice) is stable.
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 2. Load OmniScore

The OmniScore checkpoint ships a custom `ScorePredictorModel` class, so we need
`trust_remote_code=True`. The score names and output range come from the model
config — we don't hardcode them.

In [ ]:
REPO_ID = "QCRI/OmniScore-deberta-v3"
MAX_LEN = 512  # the model's own max sequence length

tokenizer = AutoTokenizer.from_pretrained(REPO_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(REPO_ID, trust_remote_code=True).to(DEVICE).eval()

SCORE_NAMES = list(model.config.score_names)
print("Score dimensions:", SCORE_NAMES)
print("Output range: [1, 5] (sigmoid-scaled in the regression head)")

## 3. Build / load the XSAMSum slice

XSAMSum is the Chinese half of XMediaSum's translations of SAMSum, used in
ClidSum (Wang et al., EMNLP 2022). The columns we need:

- `dialogue_en` — English source dialogue (utterances joined by newlines)
- `summary_zh_ref` — Chinese gold-reference summary
- `summary_zh_cand` — Chinese candidate summary produced by some system

You'll typically load this from your own files (e.g. system outputs from a
fine-tuned mBART or NLLB pipeline). To make the notebook runnable end-to-end
without external dependencies, we **also include a tiny built-in mock set**
of three examples that mimics the schema. Replace `load_dataset_slice()`
with whatever loader matches your actual files.

In [ ]:
from enum import Enum

@dataclass
class GeneralResult:
    test_index: str
    dialogue: str
    summary: str
    summary_zh: str
    predicted_en: str
    predicted_zh: str

@dataclass
class AgenticResult:
    id: str
    test_index: str
    dialogue: str
    reference_english_summary: str
    reference_chinese_summary: str
    final_summary: str
    pipeline: str
    model: str
    num_model_calls: str

# The baseline and agentic model results are formatted differently.
# Use BASELINE for baseline model results, and AGENTIC for agentic model results.
class ResultType(Enum):
    BASELINE = GeneralResult
    AGENTIC = AgenticResult

# Read a JSON with format matching the dataset_type into a list of GeneralResult
def load_dataset_slice(dataset_type, path, limit=None):
    items = []
    with open(path, "r", encoding="utf-8") as f:
      if (dataset_type == ResultType.BASELINE):
        json_items = json.load(f)
        for json_item in json_items:
          if not "predicted_en" in json_item:
            json_item["predicted_en"] = ""
          general_result = GeneralResult(**json_item)
          items.append(general_result)
      else:
        for line in f:
          json_item = json.loads(line)
          agentic_result = AgenticResult(**json_item)
          general_result = GeneralResult(
              test_index = agentic_result.test_index,
              dialogue = agentic_result.dialogue,
              summary = agentic_result.reference_english_summary,
              summary_zh = agentic_result.reference_chinese_summary,
              predicted_en = "", # No predicted English summary for agentic model
              predicted_zh = agentic_result.final_summary
          )
          items.append(general_result)
    if limit is not None:
        items = items[:limit]
    return items

# Point this at your actual XSAMSum slice when you have one.
EXAMPLES = load_dataset_slice(ResultType.AGENTIC, path="sample_data/direct_qwen_50samples.jsonl", limit=None)
print(f"Loaded {len(EXAMPLES)} examples.")
EXAMPLES[0]

## 4. Format inputs for OmniScore

OmniScore expects a **single flat text string** following this schema (from the
official model card):

```
Task: <task_name>
Source: <source text, if available>
Reference: <reference text, if available>
Candidate: <model output being evaluated>
```

For our two scoring modes:

- **Source-grounded**: include `Task` + `Source` (English dialogue) + `Candidate` (Chinese summary). Omit `Reference`.
- **Reference-based**: include `Task` + `Reference` (Chinese gold) + `Candidate` (Chinese summary). Omit `Source`.

We use `Task: summarization` since the candidate is a summary; the model card
lists `Summarization evaluation` as one of the supported tasks.

The 512-token limit will bite on long English dialogues. We don't manually
truncate — we let the tokenizer do it (`truncation=True`) but record how often
truncation happens so it's reported alongside the scores.

In [ ]:
TASK_TAG = "summarization"

def format_source_grounded(ex):
    return (
        f"Task: {TASK_TAG}\n"
        f"Source: {ex.dialogue}\n"
        f"Candidate: {ex.predicted_zh}"
    )

def format_reference_based(ex):
    return (
        f"Task: {TASK_TAG}\n"
        f"Reference: {ex.summary_zh}\n"
        f"Candidate: {ex.predicted_zh}"
    )

# Sanity check on the first example
print("=== Source-grounded input ===")
print(format_source_grounded(EXAMPLES[0])[:400], "...")
print()
print("=== Reference-based input ===")
print(format_reference_based(EXAMPLES[0])[:400], "...")

## 5. Run scoring (batched)

We score in mini-batches and:

1. Track per-example truncation (was the input clipped at 512 tokens?).
2. Run the model with `torch.no_grad()` and `.eval()` mode (already set).
3. Pull predictions from `outputs.predictions`, which has shape
   `(batch, n_score_dims)` and lives in `[1, 5]`.
4. Map each column back to its score name using `model.config.score_names`.

In [ ]:
BATCH_SIZE = 8  # safe default for a 0.2B model on a single GPU; tune for your hardware

def _was_truncated(text):
    # Did the tokenizer cut this input at MAX_LEN?
    ids = tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"]
    return len(ids) > MAX_LEN

def score_batch(texts):
    # Score a list of inputs. Returns (scores_per_input, truncation_flags).
    trunc_flags = [_was_truncated(t) for t in texts]
    batch = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    with torch.no_grad():
        out = model(**batch)
    preds = out.predictions.detach().cpu()  # shape (B, len(SCORE_NAMES))
    rows = []
    for row in preds:
        rows.append({name: float(row[i]) for i, name in enumerate(SCORE_NAMES)})
    return rows, trunc_flags

def score_all(examples, formatter, mode_name):
    # Run scoring for every example using the given input formatter.
    all_rows = []
    n = len(examples)
    t0 = time.time()
    for start in tqdm(range(0, n, BATCH_SIZE), desc=f"OmniScore [{mode_name}]"):
        chunk = examples[start : start + BATCH_SIZE]
        texts = [formatter(ex) for ex in chunk]
        scores, trunc_flags = score_batch(texts)
        for ex, sc, tr in zip(chunk, scores, trunc_flags):
            row = {"example_id": ex.test_index, "mode": mode_name, "truncated": tr}
            row.update(sc)
            all_rows.append(row)
    elapsed = time.time() - t0
    print(f"[{mode_name}] scored {n} examples in {elapsed:.1f}s "
          f"({n / max(elapsed, 1e-9):.1f} ex/s)")
    return pd.DataFrame(all_rows)

df_source = score_all(EXAMPLES, format_source_grounded, "source_grounded")
df_ref    = score_all(EXAMPLES, format_reference_based, "reference_based")
df_all = pd.concat([df_source, df_ref], ignore_index=True)
df_all

## 6. Aggregate and report

Corpus-level numbers: mean and standard deviation per score dimension per mode,
plus the share of examples that were truncated. Since OmniScore outputs are in
`[1, 5]`, a useful sanity floor for the candidate quality is around 3 — anything
substantially below that on faithfulness for source-grounded scoring is a red
flag for hallucination.

In [ ]:
def summarize(df):
    g = df.groupby("mode")
    means = g[SCORE_NAMES].mean().add_suffix("_mean")
    stds  = g[SCORE_NAMES].std().add_suffix("_std")
    trunc = g["truncated"].mean().rename("truncation_rate").to_frame()
    count = g.size().rename("n").to_frame()
    out = pd.concat([count, trunc, means, stds], axis=1)
    # Reorder: count, trunc, then mean/std interleaved per dim
    cols = ["n", "truncation_rate"]
    for s in SCORE_NAMES:
        cols.extend([f"{s}_mean", f"{s}_std"])
    return out[cols]

summary = summarize(df_all)
summary

In [ ]:
# Per-example view — easier to inspect outliers (e.g. low faithfulness scores)
view = df_all.pivot_table(
    index="example_id",
    columns="mode",
    values=SCORE_NAMES,
    aggfunc="first",
)
# Flatten the column MultiIndex: (score, mode) -> "score__mode"
view.columns = [f"{s}__{m}" for s, m in view.columns]
view = view.reset_index()
view

In [ ]:
# Flag examples worth eyeballing: low faithfulness in source-grounded mode
LOW_FAITH_THRESHOLD = 3.0
flagged = df_source[df_source["faithfulness"] < LOW_FAITH_THRESHOLD].copy()
if len(flagged):
    print(f"{len(flagged)} example(s) below faithfulness {LOW_FAITH_THRESHOLD} "
          f"in source-grounded mode — likely hallucination candidates:")
    print(flagged[["example_id", "faithfulness", "informativeness", "clarity", "plausibility"]])
else:
    print(f"No examples below faithfulness {LOW_FAITH_THRESHOLD} in source-grounded mode.")

## 7. Save results

We dump three artifacts:

- `omniscore_per_example.csv` — every score for every example in both modes
- `omniscore_summary.csv` — corpus-level means/stds per mode
- `omniscore_run_meta.json` — model id, n examples, device, score dimensions, timestamp

Together these make the run reproducible and easy to drop into a later
comparison table (alongside ROUGE / BERTScore / human study numbers).

In [ ]:
from datetime import datetime, timezone

OUT_DIR = Path("./omniscore_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

per_example_path = OUT_DIR / "mbart_gold_50_omniscore_per_example.csv"
summary_path     = OUT_DIR / "mbart_gold_50_omniscore_summary.csv"
meta_path        = OUT_DIR / "mbart_gold_50_omniscore_run_meta.json"

df_all.to_csv(per_example_path, index=False, encoding="utf-8")
summary.to_csv(summary_path, encoding="utf-8")

meta = {
    "model": REPO_ID,
    "max_seq_len": MAX_LEN,
    "score_names": SCORE_NAMES,
    "score_range": [1.0, 5.0],
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
    "n_examples": len(EXAMPLES),
    "modes": ["source_grounded", "reference_based"],
    "task_tag": TASK_TAG,
    "seed": SEED,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

print("Wrote:")
for p in (per_example_path, summary_path, meta_path):
    print(f"  {p}  ({p.stat().st_size} bytes)")

## Notes for your ClidSum write-up

A few things to keep in mind when reporting these numbers:

- **OmniScore is reference-flexible.** The same model handles both source-grounded
  and reference-based settings; the difference is only in the formatted input.
  Reporting both modes separately is the cleanest framing.
- **Score units are unfamiliar.** Unlike ROUGE (`[0, 1]`, F1-flavored) or BERTScore
  (`[0, 1]`, cosine-based after rescaling), OmniScore lives on a `[1, 5]` quality
  scale that mimics a human rubric. Don't average across metrics with different
  scales.
- **No direct human-study baseline on XSAMSum.** ClidSum's Section 6.3 human study
  is XMediaSum40k En→Zh only. Treat OmniScore on XSAMSum as a *new* automatic
  signal, not as something validated against ClidSum's annotators.
- **Truncation matters.** If `truncation_rate` is high (say >10%), report it; long
  SAMSum dialogues silently losing tail content will pull faithfulness down.
- **Try both checkpoints.** There's also `QCRI/OmniScore-mxbai` (0.3B, mxbai-large-v1
  backbone). Same interface — swap `REPO_ID` and re-run the notebook to check
  whether your conclusions are checkpoint-stable.